<a href="https://colab.research.google.com/github/andreysenna/relatoriopendulosimples/blob/main/Relat%C3%B3rio_Experimento_Pendulo_Simples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Bibliotecas utilizadas.

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

2. Leitura dos arquivos.

In [24]:
!git clone https://github.com/andreysenna/relatoriopendulosimples.git
%cd REPO

fatal: destination path 'relatoriopendulosimples' already exists and is not an empty directory.
[Errno 2] No such file or directory: 'REPO'
/content


In [25]:
# Criando os dataframes com base nos dados das tabelas do Excel
df1 = pd.read_excel('dados/100 cm.xlsx')
df2 = pd.read_excel('dados/90 cm.xlsx')
df3 = pd.read_excel('dados/80 cm.xlsx')
df4 = pd.read_excel('dados/70 cm.xlsx')

print('Primeiras 5 linhas de df1:')
display(df1.head())
print('\nPrimeiras 5 linhas de df2:')
display(df2.head())
print('\nPrimeiras 5 linhas de df3:')
display(df3.head())
print('\nPrimeiras 5 linhas de df4:')
display(df4.head())

FileNotFoundError: [Errno 2] No such file or directory: 'dados/100 cm.xlsx'

3. Cálculo dos tempos médios e períodos.

In [ ]:
# tempos de 10 oscilações: linhas = 5°, 10°, 20°, 30°, 40° e colunas = T1 a T5
t100 = df1.iloc[2:7, 2:7].astype(float)
t90 = df2.iloc[2:7, 2:7].astype(float)
t80 = df3.iloc[2:7, 2:7].astype(float)
t70 = df4.iloc[2:7, 2:7].astype(float)

alturas = [100, 90, 80, 70]
tabelas = [t100, t90, t80, t70]
angulos = ['5°', '10°', '20°', '30°', '40°']

linhas = []

for i in range(4):
    for j in range(5):
        tempos = tabelas[i].iloc[j].values
        media = np.mean(tempos)
        desvio = np.std(tempos, ddof=1)      # desvio-padrão das 5 medidas
        erro = desvio / np.sqrt(5)           # erro padrão da média
        periodo = media / 10                 # 10 oscilações
        linhas.append([alturas[i], angulos[j], media, desvio, erro, periodo, periodo**2])

colunas = ['Altura (cm)', 'Ângulo', 'Tempo Médio', 'Desvio-padrão', 'Erro padrão',
           'Período', 'Período ao Quadrado']
df_resumo = pd.DataFrame(linhas, columns=colunas)

#Agora os dataframes serão organizados por ângulo.

df_5graus = df_resumo[df_resumo['Ângulo'] == '5°']
df_10graus = df_resumo[df_resumo['Ângulo'] == '10°']
df_20graus = df_resumo[df_resumo['Ângulo'] == '20°']
df_30graus = df_resumo[df_resumo['Ângulo'] == '30°']
df_40graus = df_resumo[df_resumo['Ângulo'] == '40°']

print('Dados para 5°')
print(df_5graus)
print('Dados para 10°')
print(df_10graus)
print('Dados para 20°')
print(df_20graus)
print('Dados para 30°')
print(df_30graus)
print('Dados para 40°')
print(df_40graus)

6. Ajuste de mínimos quadrados (só 5° e 10°)

In [ ]:
# coluna com o ângulo como número
df_resumo['Ângulo (graus)'] = df_resumo['Ângulo'].str.replace('°', '').astype(int)

alturas = [100, 90, 80, 70]
cores = ['blue', 'green', 'orange', 'purple']

plt.figure(figsize=(8, 6))

for i in range(4):
    dados = df_resumo[df_resumo['Altura (cm)'] == alturas[i]]

    x = dados['Ângulo (graus)']
    y = dados['Período']

    # correlação, covariância e coeficientes da reta
    correlacao = x.corr(y)
    covariancia = x.cov(y)
    beta_1 = covariancia / x.var()
    beta_0 = y.mean() - beta_1 * x.mean()

    # incerteza da inclinação
    erros = y - (beta_0 + beta_1 * x)
    s_y = np.sqrt((erros**2).sum() / (len(x) - 2))
    erro_beta_1 = s_y / np.sqrt(((x - x.mean())**2).sum())

    print(f'Altura {alturas[i]} cm:')
    print(f'  T = {beta_0:.4f} + {beta_1:.5f} x θ')
    print(f'  inclinação = {beta_1:.5f} ± {erro_beta_1:.5f} s/grau')
    print(f'  correlação = {correlacao:.4f}')

     # Construção do gráfico
    plt.scatter(x, y, color=cores[i])
    x_vals = np.array([0, 45])
    plt.plot(x_vals, beta_0 + beta_1 * x_vals, color=cores[i], label=f'{alturas[i]} cm')

# Títulos e exibição
plt.xlabel('Ângulo de lançamento (graus)')
plt.ylabel('Período T (s)')
plt.title('Período vs Ângulo para cada altura')
plt.legend()
plt.grid()
plt.show()